# Initializing GEE Python API

In [1]:
#import packages to run GEE python API

import ee
import geemap
import pprint
import ipyleaflet
import time

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library
ee.Initialize(project='ee-diptibaral21')


In [2]:
#load packages
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import interp1d

# STEP 0: Setup and Counties (GEE workflow)

In [3]:

# From TIGER2018 filter for 9 counties in sacramento valley
counties = ee.FeatureCollection("TIGER/2018/Counties")
selected_counties = ['Butte','Colusa','Glenn','Placer','Sacramento','San Joaquin','Sutter','Yolo','Yuba']

sac_valley = counties.filter(ee.Filter.eq('STATEFP', '06')) \
                     .filter(ee.Filter.inList('NAME', selected_counties))

In [4]:
#visualizing Sac Valley Counties

Map = geemap.Map()
Map.centerObject(sac_valley, 8)  

# Add the layer
Map.addLayer(sac_valley, {},'Sacramento Valley Counties')
Map

Map(center=[39.00285080659931, -121.59337946044376], controls=(WidgetControl(options=['position', 'transparent…

# STEP 1: Rice Masking (GEE workflow)

In [5]:
# helper function to mask rice -- rice is represented by cropland value 3

def mask_rice(image):
    rice = image.select('cropland').eq(3)
    return image.updateMask(rice) \
    .clip(sac_valley) \
    .unmask(0) \
    .set('system:time_start', image.get('system:time_start'))

# hepler function to make rice frequency 
# all those pixels that were once a rice growing area in last 45 years is counted as rice growing areas 
def get_rice_frequency(start_year, end_year):
    cdl = ee.ImageCollection("USDA/NASS/CDL") \
            .filterDate(f'{start_year}-01-01', f'{end_year}-12-31') \
            .map(lambda img: img.select(['cropland'])) \
            .map(mask_rice)

    rice_sum = cdl.reduce(ee.Reducer.sum())
    rice_frequency = rice_sum.gt(0)
    return rice_sum, rice_frequency
    
rice_sum, rice_frequency = get_rice_frequency(1997, 2023)   
Map.addLayer(rice_sum.clip(sac_valley),{'min':0, 'max': 27, 'palette': ['#e5f5e0', '#a1d99b', '#31a354']},'Rice Frequency (Years)')
Map

Map(bottom=25346.0, center=[39.00285080659931, -121.59337946044376], controls=(WidgetControl(options=['positio…

In [6]:
# Get rice growing area


# pixel area in square meters
pixel_area = ee.Image.pixelArea()

# rice area image
rice_area_image = pixel_area.updateMask(rice_frequency)

# sum rice area inside each county
county_rice_area = rice_area_image.reduceRegions(
    collection=sac_valley,
    reducer=ee.Reducer.sum(),
    scale=30
)

# convert square meters to hectares and keep only county + rice_area_ha
def add_area_ha(feature):
    area_ha = ee.Number(feature.get('sum')).divide(10000)
    return ee.Feature(None, {
        'county': feature.get('NAME'),
        'rice_area_ha': area_ha
    })

county_rice_area = county_rice_area.map(add_area_ha)

#convert to pandas

df_area = geemap.ee_to_df(county_rice_area)

#save
save_path = '/group/moniergrp/dbaral/run_project/input_data'
df_area.to_csv(f"{save_path}/county_rice_area_static.csv")

# STEP 2: Processing gridMET Data (GEE workflow)

In [7]:
def load_preprocess_gridMET(start_date, end_date):
    # filtering for tmmn and tmmx
    gridMET = ee.ImageCollection("IDAHO_EPSCOR/GRIDMET") \
                 .filterDate(start_date, end_date) \
                 .select(['tmmx','tmmn'])
    # converting K to C
    def convert_units(img):
        tmmxC = img.select('tmmx').subtract(273.15).rename('tmmx')
        tmmnC = img.select('tmmn').subtract(273.15).rename('tmmn')
        tmeanC = tmmxC.add(tmmnC).divide(2).rename('tmean')
        return img.addBands([tmmxC, tmmnC, tmeanC], overwrite=True) \
                  .copyProperties(img, ['system:time_start'])

    return gridMET.map(convert_units).map(lambda img: img.updateMask(rice_frequency))

# filtering the date range from 1979 to 2023
gridMET_processed = load_preprocess_gridMET('1979-01-01','2023-12-31')

first_image = gridMET_processed.first()
info = first_image.getInfo()
#pprint.pprint(info)

In [ ]:
# visualization of gridMET
Map.addLayer(
    gridMET_processed,
     {},
    'gridMET example image (Years)'
    )
#Map.split_map(left_layer='rice_frequency', right_layer='rice_sum')
Map

# STEP 3: Processing USDA Progress Data (Python workflow)
### Interpolating for planting and harvesting date

In [ ]:
#Import the Weekly Progress Data from USDA NASS

file_path = "/group/moniergrp/dbaral/run_project/input_data/planting_harvest"
df = pd.read_csv(file_path + "/planting_harvest_usda.csv")
df.head()

In [ ]:
#Filter for relevant columns
cols = ["Program", "Year", "Period", "Week Ending", "Data Item", "Value"]
df_new = df.filter(cols).copy()
df_new.head()

In [ ]:
#Creating mask to filter data values for Planting and Harvest Progress from Data Item Column
mask = (df_new["Data Item"] == "RICE - PROGRESS, MEASURED IN PCT PLANTED") | (df_new['Data Item'] == 'RICE - PROGRESS, MEASURED IN PCT HARVESTED')
final_df = df_new[mask]
final_df.head()

In [ ]:
#Rename columns and Data Item
final_df = final_df.rename(columns={'Value': 'Progress'})
final_df['Data Item'] = final_df['Data Item'].str.replace('RICE - PROGRESS, MEASURED IN PCT PLANTED', 'PLANTING')
final_df['Data Item'] = final_df['Data Item'].str.replace('RICE - PROGRESS, MEASURED IN PCT HARVESTED', 'HARVEST')

#filter for Progress = 0
final_df = final_df[final_df['Progress'] != 0]
final_df.head()

In [ ]:
#filter the dataframe to find the pair of row that can be used to interpolate 50% progress value

filtered_data = []

for year in final_df['Year'].unique():
    df_year = final_df[final_df['Year'] == year]

    for data_item in ['HARVEST', 'PLANTING']:
        df_item = df_year[df_year['Data Item'] == data_item].sort_values(by='Progress')

        # Find consecutive pairs where the first value is < 50 and the second is >= 50
        progress_values = df_item['Progress'].tolist()
        week_ending_values = df_item['Week Ending'].tolist()

        found_pairs_indices = []
        for i in range(len(progress_values) - 1):
            if progress_values[i] < 50 and progress_values[i+1] >= 50:
                # Append the indices of the two rows to keep
                found_pairs_indices.append(df_item.index[i])
                found_pairs_indices.append(df_item.index[i+1])


        # Filter the original dataframe for rows that contain these progress values in the current year and data item
        if found_pairs_indices:
            filtered_data.append(final_df.loc[found_pairs_indices])

filtered_df = pd.concat(filtered_data).drop_duplicates().reset_index(drop=True)
display(filtered_df)

In [ ]:
#Change the object type to categorical dtype

filtered_df['Data Item'] = pd.Categorical(filtered_df['Data Item'], categories=['PLANTING', 'HARVEST'], ordered=True)
filtered_df = filtered_df.sort_values(by=['Year', 'Data Item']).reset_index(drop=True)
display(filtered_df)

In [ ]:


# Convert 'Week Ending' to datetime and then to DOY
filtered_df['Week Ending Date'] = pd.to_datetime(filtered_df['Week Ending'], format='%m/%d/%y')
filtered_df['DOY'] = filtered_df['Week Ending Date'].dt.dayofyear

interpolated_data = []

# Get the categorical dtype from the original DataFrame's 'Data Item' column
data_item_dtype = filtered_df['Data Item'].dtype

for year in filtered_df['Year'].unique():
    df_year = filtered_df[filtered_df['Year'] == year]

    for data_item in ['PLANTING', 'HARVEST']:
        df_item = df_year[df_year['Data Item'] == data_item].sort_values(by='Progress')

        if len(df_item) >= 2:
            # Get progress and DOY values
            progress_values = df_item['Progress'].tolist()
            doy_values = df_item['DOY'].tolist()

            # Create an interpolation function
            interp_func = interp1d(progress_values, doy_values)

            # Calculate DOY for 50% progress
            if 50 in progress_values:
              doy_50 = doy_values[progress_values.index(50)]
            elif min(progress_values) <= 50 <= max(progress_values):
              doy_50 = interp_func(50)
            else:
              doy_50 = None

            # Create a new row for 50% progress as a dictionary

            new_row = {'Year': year,
                        'Data Item': data_item,
                        'Progress': 50,
                        'DOY': int(doy_50)}
            interpolated_data.append(new_row)

# Create a DataFrame from the interpolated data, specifying the categorical dtype for 'Data Item'
interpolated_df_new_rows = pd.DataFrame(interpolated_data)
if not interpolated_df_new_rows.empty:
    interpolated_df_new_rows['Data Item'] = pd.Categorical(interpolated_df_new_rows['Data Item'], categories=data_item_dtype.categories, ordered=data_item_dtype.ordered)
else:
    # Handle the case where no interpolated data was generated
    interpolated_df_new_rows = pd.DataFrame(columns=filtered_df.columns)
    interpolated_df_new_rows['Data Item'] = interpolated_df_new_rows['Data Item'].astype(data_item_dtype)


# Concatenate the original and interpolated data
interpolated_df = pd.concat([filtered_df, interpolated_df_new_rows], ignore_index=True)

# Convert DOY back to date (handling leap years is simplified here)
# Ensure DOY is integer for date conversion format
interpolated_df['Date_50_Progress'] = pd.to_datetime(interpolated_df['Year'].astype(str) + '-' + interpolated_df['DOY'].astype(int).astype(str), format='%Y-%j', errors='coerce')

# Sort and display the result
interpolated_df = interpolated_df.sort_values(by=['Year', 'Data Item', 'DOY']).reset_index(drop=True)
display(interpolated_df.head())

In [ ]:
# Filter for 50% Progress only
interpolated_df = interpolated_df[interpolated_df["Progress"] == 50]
cols_keep = ["Year", "Data Item", "DOY", "Date_50_Progress"]
interpolated_df = interpolated_df[cols_keep]
display(interpolated_df)

interpolated_df = interpolated_df.rename(columns={
    'Year': 'year',
    "Data Item": "activity",
    "DOY": "doy",
    "Date_50_Progress": "date"
    })

interpolated_df["activity"] = interpolated_df["activity"].astype(str).str.lower()
display(interpolated_df)

In [ ]:
# Get average planting and Harvest Date for using in year before 1998
average_doy = interpolated_df.groupby('activity', observed=False)['doy'].mean()

# To convert DOY to date, we need a reference year so using  placeholder year as 2000.
#
placeholder_year = 2000
average_planting_date = pd.to_datetime(f'{placeholder_year}-{int(average_doy["planting"])}', format='%Y-%j')
average_harvest_date = pd.to_datetime(f'{placeholder_year}-{int(average_doy["harvest"])}', format='%Y-%j')


display(average_doy)
display(average_planting_date)
display(average_harvest_date)


In [ ]:
min_year = 1979
max_year = 1997
missing_years = [year for year in range(min_year, max_year + 1) if year not in interpolated_df['year'].unique()]

add_dataframe = []

for year in missing_years:
    doy_planting = int(average_doy["planting"])
    doy_harvest = int(average_doy["harvest"])

    planting_date = pd.to_datetime(f'{year}-{doy_planting}', format='%Y-%j')
    harvest_date = pd.to_datetime(f'{year}-{doy_harvest}', format='%Y-%j')

    # Add planting data
    add_dataframe.append({
        'year': year,
        'activity': 'planting',
        'doy': doy_planting,
        'date': planting_date
    })

    # Add harvest data
    add_dataframe.append({
        'year': year,
        'activity': 'harvest',
        'doy': doy_harvest,
        'date': harvest_date
    })

add_df = pd.DataFrame(add_dataframe)
display(add_df)

In [ ]:
final_df = pd.concat([add_df, interpolated_df], ignore_index=True)

# Drop duplicates based on 'year' and 'plant_harvest' before pivoting
final_df = final_df.drop_duplicates(subset=['year', 'activity']).reset_index(drop=True)

# Pivot the DataFrame to have 'planting' and 'harvest' as columns
final_df = final_df.pivot(index='year', columns='activity', values=['doy', 'date']).reset_index()

# Flatten the MultiIndex columns
final_df.columns = [f'{col[0]}_{col[1]}' if col[1] else col[0] for col in final_df.columns]

final_df = final_df[["year", "doy_planting", "date_planting", "doy_harvest", "date_harvest"]]
final_df = final_df.dropna()

display(final_df)


In [ ]:


test = final_df[final_df['year'] == 2024]
test

final_df['year'] = final_df['year'].astype(int)
final_df['doy_planting'] = final_df['doy_planting'].astype(int)
final_df['doy_harvest'] = final_df['doy_harvest'].astype(int)
final_df.info()


In [ ]:
# Convert 'year', 'doy_planting', and 'doy_harvest' to float type for plotting
final_df['year'] = final_df['year'].astype(float)
final_df['doy_planting'] = final_df['doy_planting'].astype(float)
final_df['doy_harvest'] = final_df['doy_harvest'].astype(float)

# Filter data for years after 1998
plot_df = final_df[final_df['year'] >= 1998].copy()

# Plotting
plt.figure(figsize=(12, 6))

# Plot planting dates with a trend line
sns.regplot(x='year', y='doy_planting', data=plot_df, label='Planting Date', scatter_kws={'s': 20})

# Plot harvest dates with a trend line
sns.regplot(x='year', y='doy_harvest', data=plot_df, label='Harvest Date', scatter_kws={'s': 20})

plt.xlabel('Year')
plt.ylabel('Day of Year (DOY)')
plt.title('Planting and Harvest Dates Over Time with Trend Lines (1998 onwards)')
plt.legend() # Ensure legend is displayed
plt.grid(True)
plt.show()

In [22]:

#export
export = final_df.to_csv(os.path.join(file_path, 'planting_harvest_processed.csv'), index = False)

# STEP 4: Insert planting-harvest table (GEE worklflow)

In [23]:
# Load planting-harvest table
season_table = ee.FeatureCollection('projects/ee-diptibaral21/assets/Planting_Harvest_Dates')

In [24]:
# Build growing season collection
def filter_by_season(feature, initial):
    plant_date = ee.Date(feature.get('date_planting'))
    harvest_date = ee.Date(feature.get('date_harvest'))
    year = ee.Number(feature.get('year'))

    filtered = gridMET_processed.filterDate(plant_date, harvest_date) \
                                .map(lambda img: img.set('year', year))

    return ee.ImageCollection(initial).merge(filtered)

growing_season_images = ee.ImageCollection(season_table.iterate(filter_by_season, ee.ImageCollection([])))

In [25]:
# Extract daily stats per county
def extract_daily_stats(image):
    date = ee.Date(image.get('system:time_start'))
    reduced = image.reduceRegions(collection=sac_valley,
                                  reducer=ee.Reducer.mean(),
                                  scale=4000,
                                  crs='EPSG:3310')

    def set_props(f):
        return f.set({
            'date': date.format('YYYY-MM-dd'),
            'year': date.get('year'),
            'month': date.get('month'),
            'day': date.get('day')
        })

    return reduced.map(set_props)

daily_stats = growing_season_images.map(extract_daily_stats).flatten()

In [27]:
# Format results for export
formatted_stats = daily_stats.map(lambda f: ee.Feature(None, {
    'county': f.get('NAME'),
    'date': f.get('date'),
    'year': f.get('year'),
    'month': f.get('month'),
    'day': f.get('day'),
    'tmmn': f.get('tmmn'),
    'tmmx': f.get('tmmx'),
    'tmean': f.get('tmean')
}))

In [28]:
# Export 
task = ee.batch.Export.table.toDrive(
    collection=formatted_stats,
    description='Daily_Rice_Temperature_1979_2023',
    fileFormat='CSV',
    selectors=['county', 'date', 'year', 'month', 'day', 'tmmn', 'tmmx', 'tmean']
)
task.start()



# STEP 5: Phenology Detection Using Growing Degree Day Model (Python Workflow)

In [ ]:
#import daily temp data

file_path = "/group/moniergrp/dbaral/run_project/input_data/gridmet"

df = pd.read_csv(file_path + "/Daily_Rice_Temperature_1979_2023_v2.csv")
df = df.sort_values(by = ['county', 'year', 'month', 'day'])

print(df.head())
print(df.shape)

In [ ]:
#Checking for complete data in df

expected_years = range(1979, 2024)
expected_months = range(4, 11)
expected_days = range(1, 32)

counties = df['county'].unique()
years = df['year'].unique()
months = df['month'].unique()
days = df['day'].unique()

print("Checking data coverage:")
print(f"Expected_years: {list(expected_years)}")
print(f"Years in data: {years}")
print(f"Expected_months: {list(expected_months)}")
print(f"Months in data: {list(months)}")
print(f"Expected_days: {list(expected_days)}")
print(f"Days in data: {days}")

#Checking if each county has all years, months and days and if there are any missing values

for county in counties:
    county_data = df[df['county'] == county]
    county_years = county_data['year'].unique()
    county_months = county_data['month'].unique()
    county_days = county_data['day'].unique()
    print(f"\nCounty: {county}")
    print(f"Years in data: {county_years}")
    print(f"Months in data: {county_months}")
    print(f"Days in data: {county_days}")

    missing_values = county_data.isnull().sum()
    if missing_values.sum() > 0:
        print(f"\nMissing values for {county}:")
        print(missing_values[missing_values > 0])
    else:
        print(f"\nNo missing values for {county}")



In [31]:
# base temp, lower temp, optimum temp and threshold gdd for each rice stage: needed to calculated daily gdd
#temp are in C
#referenced from Sharifi et al 2016 - for medium grain rice
# Pl_PI: Planting(PL) to Panicle Initiation(PI)
#PI_HD: Panicle Initiation (PI) to Heading (HD)
#HD_R7: Heading(HD) to R7 stage

growth_stages = {
    'PL_PI' : {
       'tbase': 10,
       'tlower': 14.30,
       'topt': 27.73,
       'threshold': 454
   },
   'PI_HD' : {
       'tbase': 9.8,
       'tlower': 15.37,
       'topt': 30.73,
       'threshold': 356
   },
   'HD_R7' : {
       'tbase': 8.9,
       'tlower': 15.93,
       'topt': 33.03,
       'threshold': 203
   }
}

In [32]:
#function to calculate daily gdd
def calculate_daily_gdd (tmmn, tmmx, tbase, tlower, topt):
  tmmx_adj = min(tmmx, topt)
  tmmn_adj = min(tmmn, tlower)
  tmean_adj = (tmmx_adj + tmmn_adj)/2
  gdd = max(tmean_adj - tbase, 0)
  return gdd

In [ ]:
# Test cases for calculate_daily_gdd function

# Test case 1: Tmin and Tmax within thresholds
tmmn_test1, tmmx_test1, tbase_test1, tlower_test1, topt_test1 = 10, 30, 10, 14, 20
gdd_test1 = calculate_daily_gdd(tmmn_test1, tmmx_test1, tbase_test1, tlower_test1, topt_test1)
print(f"Test Case 1: tmmn={tmmn_test1}, tmmx={tmmx_test1}, tbase={tbase_test1}, tlower={tlower_test1}, topt={topt_test1} -> GDD = {gdd_test1}")

# Test case 2: Tmin below tbase
tmmn_test2, tmmx_test2, tbase_test2, tlower_test2, topt_test2 = 5, 20, 10, 14, 30
gdd_test2 = calculate_daily_gdd(tmmn_test2, tmmx_test2, tbase_test2, tlower_test2, topt_test2)
print(f"Test Case 2: tmmn={tmmn_test2}, tmmx={tmmx_test2}, tbase={tbase_test2}, tlower={tlower_test2}, topt={topt_test2} -> GDD = {gdd_test2}")

# Test case 3: Tmax above topt
tmmn_test3, tmmx_test3, tbase_test3, tlower_test3, topt_test3 = 15, 35, 10, 14, 30
gdd_test3 = calculate_daily_gdd(tmmn_test3, tmmx_test3, tbase_test3, tlower_test3, topt_test3)
print(f"Test Case 3: tmmn={tmmn_test3}, tmmx={tmmx_test3}, tbase={tbase_test3}, tlower={tlower_test3}, topt={topt_test3} -> GDD = {gdd_test3}")

# Test case 4: Tmin above tlower
tmmn_test4, tmmx_test4, tbase_test4, tlower_test4, topt_test4 = 18, 25, 10, 14, 30
gdd_test4 = calculate_daily_gdd(tmmn_test4, tmmx_test4, tbase_test4, tlower_test4, topt_test4)
print(f"Test Case 4: tmmn={tmmn_test4}, tmmx={tmmx_test4}, tbase={tbase_test4}, tlower={tlower_test4}, topt={topt_test4} -> GDD = {gdd_test4}")

# Test case 5: Both Tmin and Tmax outside thresholds
tmmn_test5, tmmx_test5, tbase_test5, tlower_test5, topt_test5 = 8, 40, 10, 14, 30
gdd_test5 = calculate_daily_gdd(tmmn_test5, tmmx_test5, tbase_test5, tlower_test5, topt_test5)
print(f"Test Case 5: tmmn={tmmn_test5}, tmmx={tmmx_test5}, tbase={tbase_test5}, tlower={tlower_test5}, topt={topt_test5} -> GDD = {gdd_test5}")

# Test case 6: Tmean below tbase
tmmn_test6, tmmx_test6, tbase_test6, tlower_test6, topt_test6 = 5, 10, 10, 14, 30
gdd_test6 = calculate_daily_gdd(tmmn_test6, tmmx_test6, tbase_test6, tlower_test6, topt_test6)
print(f"Test Case 6: tmmn={tmmn_test6}, tmmx={tmmx_test6}, tbase={tbase_test6}, tlower={tlower_test6}, topt={topt_test6} -> GDD = {gdd_test6}")

In [34]:
#Initialize results storage

results = []   #Remember this has to be outside the loop and should be done before starting the for loop

#process each county and year combination

for (county, year), group in df.groupby(['county', 'year']):
  season_data = group.copy()
  season_data = season_data.sort_values(by= 'date')

  #Convert 'date' column to datetime objects for reliable comparison
  season_data['date'] = pd.to_datetime(season_data['date'])

  # Use the first date in the 'date' column as planting date
  planting_date = season_data['date'].iloc[0] if not season_data.empty else None
  # Use the last date in the 'date' column as harvest date
  harvest_date = season_data['date'].iloc[-1] if not season_data.empty else None


  # Filtering data from the planting date onwards
  season_data = season_data[(season_data['date'] >= planting_date)].copy() # Use .copy() to avoid SettingWithCopyWarning

  #print(season_data.head())
  #print(season_data.info)
  #print(season_data.describe())

  #Initialize growth tracking
  current_stage = 'PL_PI'
  stage_gdd = 0
  accumulated_gdd = 0
  transition_dates = {
      'pl_date': season_data.iloc[0]['date'] if not season_data.empty else None, # To handle empty season_data
      'pi_date': None,
      'hd_date': None,
      'r7_date': None,
      'grain_fill_start_date': None,
      'grain_fill_end_date': None
  }

  # Handling case where season_data might be empty after filtering
  if season_data.empty:
    print(f"No data for growing season in {county}, {year}")
    continue # Skip to the next group if no data for the season


  daily_results = []

  for index, row in season_data.iterrows():
      # Calculate daily GDD based on the current stage's parameters
      stage_params = growth_stages.get(current_stage)
      daily_gdd = 0

      if stage_params:
          daily_gdd = calculate_daily_gdd(
              row['tmmn'],
              row['tmmx'],
              stage_params['tbase'],
              stage_params['tlower'],
              stage_params['topt']
          )
          stage_gdd += daily_gdd
          accumulated_gdd += daily_gdd


      # Check for stage transition based on GDD
      new_stage = None
      if current_stage in growth_stages:
          stage_params = growth_stages.get(current_stage)
          if stage_params and stage_gdd >= stage_params['threshold']:
              if current_stage == 'PL_PI':
                  transition_dates['pi_date'] = row['date']
                  new_stage = 'PI_HD'
              elif current_stage == 'PI_HD':
                  transition_dates['hd_date'] = row['date']
                  new_stage = 'HD_R7'
              elif current_stage == 'HD_R7':
                  transition_dates['r7_date'] = row['date']
                  new_stage = 'completed' # Transition to a completed stage

      # Update current stage and reset stage_gdd if a transition occurred
      if new_stage:
          current_stage = new_stage
          stage_gdd = 0 # Reset stage_gdd for the new stage

      # Check for grain fill start date based on hd_date + 0 days
      if current_stage == 'completed' and transition_dates['hd_date'] is not None and transition_dates['grain_fill_start_date'] is None:
          transition_dates['grain_fill_start_date'] = transition_dates['hd_date']
          transition_dates['grain_fill_end_date'] = transition_dates['grain_fill_start_date'] + pd.Timedelta(days=30)


      # If in the grain fill period, update the stage
      if transition_dates['grain_fill_start_date'] is not None and transition_dates['grain_fill_end_date'] is not None:
          if row['date'] >= transition_dates['grain_fill_start_date'] and row['date'] <= transition_dates['grain_fill_end_date']:
              current_stage = 'grain_fill'

      # Only append results if the date is within the extended season
      if row['date'] <= harvest_date:
          # Add to overall results
          daily_results.append({
              'county': county,
              'date': row['date'],
              'year': year,
              'month': row['month'],
              'day': row['day'],
              'stage': current_stage, # Use current_stage after transition check
              'daily_gdd': daily_gdd,
              'accumulated_gdd': accumulated_gdd,
              'stage_gdd': stage_gdd, # stage_gdd is reset upon transition
              'pl_date': transition_dates['pl_date'],
              'pi_date': transition_dates['pi_date'],
              'hd_date': transition_dates['hd_date'],
              'r7_date': transition_dates['r7_date'],
              'grain_fill_start_date': transition_dates['grain_fill_start_date'],
              'grain_fill_end_date': transition_dates['grain_fill_end_date']
          })
  results.extend(daily_results)

In [ ]:
results_df = pd.DataFrame(results)
print(results_df.head())
print(results_df.shape)

In [ ]:
print(f"Results exported for {len(df['county'].unique())} counties and {len(df['year'].unique())} years.")


In [ ]:
#Converting date cols to datetime format

date_cols = ['pl_date', 'pi_date', 'hd_date', 'r7_date', 'grain_fill_start_date', 'grain_fill_end_date']
results_df[date_cols] = results_df[date_cols].apply(pd.to_datetime)
# print(results_df.dtypes)

#calculating day of the year for each of the date cols
for col in date_cols:
    results_df[f'{col}_doy'] = results_df[col].dt.dayofyear
# print(results_df.head)

#converting date column to datetime format
results_df['date'] = pd.to_datetime(results_df['date'])
print(results_df.dtypes)

# Calculate days after planting for pi_date and hd_date
results_df['pi_dap'] = (results_df['pi_date'] - results_df['pl_date']).dt.days
results_df['hd_dap'] = (results_df['hd_date'] - results_df['pl_date']).dt.days
results_df['r7_dap'] = (results_df['r7_date'] - results_df['pl_date']).dt.days
results_df['grain_fill_dap'] = (results_df['grain_fill_start_date'] - results_df['pl_date']).dt.days
print(results_df.head)

results_df.to_csv(file_path + '/Rice_Growth_Stages_GDD_Results_1979_2023_v2.csv', index=False)

In [ ]:
#creating new data frame to extract transition dates only

transition_dates = results_df.groupby(['county', 'year']).agg({
    'pl_date': 'first',
    'pi_date': 'first',
    'hd_date': 'first',
    'r7_date': 'first',
    'grain_fill_start_date': 'first',
    'grain_fill_end_date': 'first',
    'pl_date_doy': 'first',
    'pi_date_doy': 'first',
    'hd_date_doy': 'first',
    'r7_date_doy': 'first',
    'grain_fill_start_date_doy': 'first',
    'grain_fill_end_date_doy': 'first',
    'pi_dap': 'first',
    'hd_dap': 'first',
    'r7_dap': 'first',
    'grain_fill_dap': 'first'
}).reset_index()

transition_dates.to_csv(file_path + '/Phenology_Transition_1979_2023_v2.csv', index=False)

In [ ]:
#merge temp and gdd data to work for stress indices because we can identify the stages: Booting and Flowering only from gdd
#and to calulate stres indices we need daily temp data

temp_df = df
gdd_df = results_df

#changing date column to pd datetime format for temp data
temp_df['date'] = pd.to_datetime(temp_df['date'])

temp_gdd_df = pd.merge(
    left = temp_df,
    right = gdd_df,
    on =  ['county', 'year', 'month', 'day', 'date'],
    how = 'right'
)

temp_gdd_df.to_csv(file_path +'/Combined_Temp_GDD_1979_2024_v2.csv', index=False)


In [ ]:
#Defining booting and flowering stage
##Booting is 7 days after Panicle Initiation and until 7 days before 50% heading
##Flowering is 7 days before 50% heading and 7 days after 50% heading
##Grainfill period is 30 days following 50%heading

results = []

grouped = temp_gdd_df.groupby(['county', 'year'])

#Loop through each group
for i, ((county, year), group) in enumerate(grouped):
    group = group.copy() # Get the group DataFrame explicitly
    #print(f"Group {i}: {county}, {year}, {len(group)} rows")
    #print(group.dtypes)
    #get first dates for this group
    first_pi_date = group['pi_date'].dropna().iloc[0]
    first_hd_date = group['hd_date'].dropna().iloc[0]
    #print(first_pi_date)
    #print(first_hd_date)

    booting_start = first_pi_date + pd.Timedelta(days = 7)
    booting_end = first_hd_date - pd.Timedelta(days = 7)

    flowering_start = first_hd_date - pd.Timedelta(days = 7)
    flowering_end = first_hd_date + pd.Timedelta(days = 7)

    grain_fill_start = first_hd_date
    grain_fill_end = first_hd_date + pd.Timedelta(days = 30)

    # Initialize the 'stage' column with a default value

    group['stage'] = 'growth stage' # Default stage

    #filtering for booting stage and naming the stage as booting

    group.loc[
        (group['date'] >= booting_start) &
        (group['date'] <= booting_end),
        'stage'
    ] = 'booting'

    #filtering for flowering stage and naming the stage column as flowering
    flowering_mask = (
        (group['date'] >= flowering_start) &
        (group['date'] <= first_hd_date)
    )

    group.loc[flowering_mask, 'stage'] = 'flowering'

    #filtering for overlapping period of flowering and grain fill stage and naming it flowering and grainfill

    flowering_grainfill_mask = (
        (group["date"] > first_hd_date) &
        (group["date"] <= flowering_end)
    )

    group.loc[flowering_grainfill_mask, 'stage'] = 'flowering_grainfill'


    #filtering for grain_fil period and naming it as grainfill

    grain_fill_mask = (
        (group['date'] > flowering_end) &
        (group['date'] <= grain_fill_end)
    )

    group.loc[grain_fill_mask, 'stage'] = 'grain_fill'


    # Debugging print statements for flowering mask
    if i == 0:
      print(f"--- Debugging Stage Assignment for {county}, {year} ---")
      print(f"Booting start date: {booting_start}, Booting end date: {booting_end}")
      print(f"Flowering start date: {flowering_start}, Flowering end date: {flowering_end}")
      print(f"Dates marked as flowering: {group.loc[flowering_mask, 'date']}")
      print(f"Dates marked as flowering and grainfill: {group.loc[flowering_grainfill_mask, 'date']}")
      print(f"Grain fill start date: {grain_fill_start}, Grain fill end date: {grain_fill_end}")
      print(f"Dates marked as grain filling: {group.loc[grain_fill_mask, 'date']}")

      print("--- End Debugging ---")


    # Add tmmx_gf and tmmn_gf columns for grain fill period
    group['tmmx_gf'] = np.nan
    group['tmmn_gf'] = np.nan

    # Calculate max and min temperature for the entire grain fill period for this county and year
    grain_fill_period_data = group[group['stage'] == 'grain_fill']
    if not grain_fill_period_data.empty:
        max_tmmx_gf = grain_fill_period_data['tmmx'].max()
        min_tmmn_gf = grain_fill_period_data['tmmn'].min()
        # Assign the calculated max and min to all rows within the grain fill period for this group
        group.loc[group['stage'] == 'grain_fill', 'tmmx_gf'] = max_tmmx_gf
        group.loc[group['stage'] == 'grain_fill', 'tmmn_gf'] = min_tmmn_gf

    results.append(group)

#combine all groups
final_results = pd.concat(results)

In [ ]:

#Extract data frame for booting, flowering and HD_R7 stage only, AND all rows where grain_fill is True

booting_flowering_grainfill_df = final_results[
    final_results['stage'].isin(['booting', 'flowering', 'flowering_grainfill', 'HD_R7', 'grain_fill'])
].copy()

booting_flowering_grainfill_df.to_csv(file_path + '/Booting_Flowering_GrainFill_1979_2023_v2.csv', index=False)

In [ ]:
#creating growth stage table
# INPUT / OUTPUT
in_csv  = "/Booting_Flowering_GrainFill_1979_2023_v2.csv"
out_csv = "/9_County_Stage_Dates_1979_2023.csv"

# How to build the 3 output stages from the daily "stage" labels in the big file
STAGE_MAP = {
    "booting":    ["booting"],
    "flowering":  ["flowering", "flowering_grainfill"],      # include overlap
    "grain_fill": ["grain_fill", "flowering_grainfill"],     # include overlap
}

df = pd.read_csv(file_path + in_csv)

# make sure date is datetime
df["date"] = pd.to_datetime(df["date"])

rows = []
for (county, year), g in df.groupby(["county", "year"], sort=True):
    for out_stage, labels in STAGE_MAP.items():
        sub = g[g["stage"].isin(labels)]
        if sub.empty:
            continue
        rows.append({
            "county": county,
            "year": int(year),
            "stage": out_stage,
            "start_date": sub["date"].min().date().isoformat(),
            "end_date": sub["date"].max().date().isoformat(),
        })

stage_dates = pd.DataFrame(rows).sort_values(["county", "year", "stage"]).reset_index(drop=True)
stage_dates.to_csv(file_path + out_csv, index=False)

print("Saved:", out_csv)
print(stage_dates.head(10))

# STEP 6: Stress Calculation (GEE Workflow)

### pixel wise stress calculation and then aggregaring over the couties

In [ ]:

# Load stage table
stage_table = ee.FeatureCollection(
    "projects/ee-diptibaral21/assets/9_County_Stage_Dates_1979_2023"
)

In [45]:

# Function: calculate daily stress for a stage

def calculate_daily_stress(img, stage_feature):
    stage = ee.String(stage_feature.get('stage'))

    booting_cold = ee.Image.constant(13)
    flowering_cold = ee.Image.constant(10.9)
    flowering_heat = ee.Image.constant(37.5)

    # Default zero images
    cdstress_bo = ee.Image.constant(0).rename('cdstress_bo')
    cdstress_fl = ee.Image.constant(0).rename('cdstress_fl')
    htstress_fl = ee.Image.constant(0).rename('htstress_fl')

    # Cold stress for Booting stage
    cdstress_bo = ee.Image(
        ee.Algorithms.If(
            stage.equals('booting'),
            img.select('tmmn').lt(booting_cold)
               .multiply(booting_cold.subtract(img.select('tmmn')))
               .rename('cdstress_bo'),
            cdstress_bo
        )
    )

    # Cold stress for Flowering stage
    cdstress_fl = ee.Image(
        ee.Algorithms.If(
            stage.equals('flowering'),
            img.select('tmmn').lt(flowering_cold)
               .multiply(flowering_cold.subtract(img.select('tmmn')))
               .rename('cdstress_fl'),
            cdstress_fl
        )
    )

    # Heat stress for Flowering stage
    htstress_fl = ee.Image(
        ee.Algorithms.If(
            stage.equals('flowering'),
            img.select('tmmx').gt(flowering_heat)
               .multiply(img.select('tmmx').subtract(flowering_heat))
               .rename('htstress_fl'),
            htstress_fl
        )
    )

    return img.select(['tmmn', 'tmmx', 'tmean']).addBands([cdstress_bo, cdstress_fl, htstress_fl])




In [ ]:
# ---------------------------------------------
# Debug daily stress
# ---------------------------------------------
first_stage_feature = stage_table.first()
example_date = ee.Date(first_stage_feature.get('start_date'))
example_temp_img = growing_season_images.filterDate(example_date, example_date.advance(1, 'day')).first()
example_daily_stress = calculate_daily_stress(example_temp_img, first_stage_feature)

print('Debug Daily Stress Function Results:', example_daily_stress.getInfo())

# Visualization
Map = geemap.Map(center=[38.5, -121.8], zoom=8)

Map.addLayer(
    example_daily_stress.select('cdstress_bo'),
    {'min': 0, 'max': 0.5, 'palette': ['white', 'yellow', 'red']},
    'Example Cold Stress Booting Viz'
)

Map.addLayer(
    example_daily_stress.select('tmmn'),
    {'min': 12, 'max': 15, 'palette': ['white', 'yellow', 'red']},
    'Example tmmn Viz'
)

Map.addLayer(
    example_daily_stress.select('tmmx'),
    {'min': 30, 'max': 33, 'palette': ['white', 'pink', 'red']},
    'Example tmmx Viz'
)
Map

In [ ]:


# Function: calculate yearly stress for one stage feature

def calculate_yearly_stage_stress(stage_feature):
    county = ee.String(stage_feature.get('county'))
    year = ee.Number(stage_feature.get('year'))
    start = ee.Date.parse('YYYY-MM-dd', stage_feature.get('start_date'))
    end = ee.Date.parse('YYYY-MM-dd', stage_feature.get('end_date'))

    stage_temps = growing_season_images.filterDate(start, end)

    # Map daily stress over images
    daily_stress = stage_temps.map(lambda img: calculate_daily_stress(img, stage_feature))

    # Mean of climate variables
    climate_means = daily_stress.select(['tmmn', 'tmmx', 'tmean']).mean()

    # Sum of stress indices
    stress_sums = daily_stress.select(['cdstress_bo', 'cdstress_fl', 'htstress_fl']).sum()

    # Combine
    yearly_stress = climate_means.addBands(stress_sums).set({
        'county': county,
        'stage': stage_feature.get('stage'),
        'year': year
    })

    # Reduce by county
    single_county = sac_valley.filter(ee.Filter.eq('NAME', county))
    return yearly_stress.reduceRegions(
        collection=single_county,
        reducer=ee.Reducer.mean(),
        scale=4000,
        crs='EPSG:3310'
    ).map(lambda f: f.set({
        'county': county,
        'stage': stage_feature.get('stage'),
        'year': year
    }))





In [50]:

# Debug yearly stage stress

debug_yearly_stage_table = stage_table \
    .filter(ee.Filter.eq('county', 'Butte')) \
    .filter(ee.Filter.eq('year', 2018)) \
    .filter(ee.Filter.eq('stage', 'flowering'))

#print("Debug Stage Table:", debug_yearly_stage_table.getInfo())

yearly_stress_flowering_butte_2018 = calculate_yearly_stage_stress(debug_yearly_stage_table.first())
#print("Yearly Stress during Flowering in Butte in 2018:", yearly_stress_flowering_butte_2018.getInfo())



In [52]:

# Calculate stress for all years

all_year_table = ee.FeatureCollection(stage_table.map(calculate_yearly_stage_stress).flatten())

all_year_table_selected = all_year_table.select([
    'county', 'year', 'stage',
    'cdstress_bo', 'cdstress_fl', 'htstress_fl'
])

first_export_feature = all_year_table_selected.first()
#first_export_feature.getInfo()


In [54]:

# Export to Google Drive

task_drive = ee.batch.Export.table.toDrive(
    collection=all_year_table_selected,
    description='AnnualStress_1979_2023_v2',
    fileFormat='CSV',
    selectors=['county', 'year', 'stage', 'cdstress_bo', 'cdstress_fl', 'htstress_fl']
)
task_drive.start()




# STEP 7: Extracting variables for each growth stages (Python Workflow)

In [55]:
file_path = "/group/moniergrp/dbaral/run_project/input_data"
df = pd.read_csv(file_path +  "/gridmet/Booting_Flowering_GrainFill_1979_2023_v2.csv",
    parse_dates=["date"]
)
planting_harvest_dates = pd.read_csv(file_path + "/planting_harvest/planting_harvest_processed.csv", parse_dates = ["date_planting", "date_harvest"])

In [56]:
def make_stage_file(df, stage_name, suffix):
    if isinstance(stage_name, str):
        mask = df["stage"].eq(stage_name)
    else:
        mask = df["stage"].isin(stage_name)

    out = (
        df.loc[mask]
        .groupby(["county", "year"], as_index=False)
        .agg(
            **{
                f"tmmn_{suffix}": ("tmmn", "mean"),
                f"tmmx_{suffix}": ("tmmx", "mean"),
                f"tmean_{suffix}": ("tmean", "mean"),
            }
        )
        .reset_index(drop=True)
    )
    return out

In [58]:

# Booting / Flowering / Grain fill

booting_df   = make_stage_file(df, "booting", "bo")
flowering_df = make_stage_file(df, ["flowering", "flowering_grainfill"], "fl")
grainfill_df = make_stage_file(df, ["grain_fill", "flowering_grainfill"], "gf")

# Save stage outputs
booting_df.to_csv(file_path + "/booting_feature_variables.csv", index=False)
flowering_df.to_csv(file_path + "/flowering_feature_variables.csv", index=False)
grainfill_df.to_csv(file_path + "/grainfill_feature_variables.csv", index=False)




In [59]:
combined_df = df.merge(planting_harvest_dates, on = "year", how = "left")

growing_season_df = combined_df[
    (combined_df["date"] >= combined_df["date_planting"]) &
    (combined_df["date"] <= combined_df["date_harvest"])
]

growing_season_features = (
    growing_season_df
    .groupby(["county", "year"], as_index=False)
    .agg(
        tmmn_gs=("tmmn", "mean"),
        tmmx_gs=("tmmx", "mean"),
        tmean_gs=("tmean", "mean"),
    )
    .sort_values(["county", "year"])
    .reset_index(drop=True)
)

growing_season_features.to_csv(
    file_path + "/gridmet/growing_season_feature_variables.csv",
    index=False
)


# STEP 8: Preparing data for model input (Python Workflow)

In [60]:
#import data
file_path = "/group/moniergrp/dbaral/run_project/input_data"
save_path = "/group/moniergrp/dbaral/run_project/input_data/gridmet_hist_model_input"

yield_df = pd.read_csv(file_path + '/yield/rice_yield_1979_2023.csv')
stress = pd.read_csv(file_path + '/gridmet/AnnualStress_1979_2023_v2.csv')
booting = pd.read_csv(file_path + '/gridmet/booting_feature_variables.csv')
flowering = pd.read_csv(file_path + '/gridmet/flowering_feature_variables.csv')
grainfill =  pd.read_csv(file_path + '/gridmet/grainfill_feature_variables.csv')
averageTemp = pd.read_csv(file_path + '/gridmet/growing_season_feature_variables.csv')

In [ ]:
print(yield_df.head())
print(stress.head())
print(booting.head())
print(flowering.head())
print(grainfill.head())
print(averageTemp.head())

In [ ]:
#Stress df has repeated years columns - each year repeating thrice for booting, flowering and grain_fill period
#group the data by couunty and year and sum the stress columns to get one column for all stress indices for one year

# Group by county and year and sum the stress columns
stress = stress.groupby(['county', 'year'])[['cdstress_bo', 'cdstress_fl', 'htstress_fl']].sum().reset_index()

print(stress.head())

In [ ]:
#Merge all datasets

model_df = pd.merge(
    left= yield_df,
    right=stress,
    how='outer',
    on=['county','year']
)

model_df = pd.merge(
    left=model_df,
    right=booting,
    how='outer',
    on=['county','year']
)

model_df = pd.merge(
    left=model_df,
    right=flowering,
    how='outer',
    on=['county','year']
)

model_df = pd.merge(
    left=model_df,
    right=grainfill,
    how='outer',
    on=['county','year']
)

lasso_data = pd.merge(
    left=model_df,
    right=averageTemp,
    how='outer',
    on=['county','year']
)
print(lasso_data.head())

export = lasso_data.to_csv(save_path + '/Lasso_Model_Input_Variables_1979_2023_v2.csv', index=False)
